In [2]:
import pandas as pd

orders = pd.read_excel("Cleaned_Orders_Dataset.xlsx")

## Setting Up the Database

Before running any queries, the cleaned dataset was loaded into a SQLite database (`orders.db`) using Python's `sqlite3` library. This allows SQL queries to be executed directly against the data using `pandas.read_sql_query()`, combining the structure of a relational database with the flexibility of a notebook environment.



In [3]:
import sqlite3

# this opens orders.db if it exists, or creates it if it doesn't
connection = sqlite3.connect("orders.db")

# this takes your dataframe (let's say it's called df) and writes it
# into the database as a table named "orders"
orders.to_sql("orders", connection, if_exists="replace", index=False)

1200

In [4]:
test_query = "SELECT * FROM orders LIMIT 5"
result = pd.read_sql_query(test_query, connection)
result

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## Section 1: Revenue Analysis

The first set of questions focuses on where revenue is concentrated, starting with which products and payment methods generate the most total revenue.



### 1. Which products generate the most total revenue?

In [5]:
query_1 = """
SELECT Product, SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC
"""

result_1 = pd.read_sql_query(query_1, connection)
result_1

,Product,TotalRevenue
0,Chair,195620.11
1,Printer,195612.61
2,Laptop,192126.56
3,Tablet,186568.95
4,Monitor,175651.41
5,Desk,167459.93
6,Phone,151722.39


In [6]:
query_1a = """
SELECT product, SUM(Quantity) AS Quantity, COUNT(OrderID) as OrderCount, AVG(Quantity) as AverageQuantity 
FROM orders 
GROUP BY product  
ORDER BY Quantity DESC
"""

result_1a = pd.read_sql_query(query_1a, connection)
result_1a

,Product,Quantity,OrderCount,AverageQuantity
0,Chair,562,178,3.157303
1,Printer,542,181,2.994475
2,Laptop,535,173,3.092486
3,Desk,508,170,2.988235
4,Tablet,497,179,2.776536
5,Monitor,480,163,2.944785
6,Phone,411,156,2.634615


**Findings:** Chair leads in total revenue not because it's ordered more frequently than other products (it actually has slightly fewer orders than Printer and Tablet that lead the table on that front), but because of a higher average quantity per order (3.16 units), the highest among all products. This suggests Chair buyers tend to purchase in larger batches per transaction rather than Chair simply being the most popular individual item.

Phone, by contrast, ranks lowest across total revenue, total quantity, order count, and has the lowest unit price of all products, suggesting it underperforms across the board rather than being held back by any single factor.

### 2. Which payment methods generate the most total revenue?

Beyond product level revenue, it's worth checking whether certain payment methods are associated with higher total spend. This could point to customer behavior patterns, for example, customers using credit cards might tend to make larger purchases than those paying through other methods.

In [7]:
query_2 = """
SELECT PaymentMethod, SUM(TotalPrice) AS TotalRevenue
FROM orders
GROUP BY PaymentMethod
ORDER BY TotalRevenue DESC
"""

result_2 = pd.read_sql_query(query_2, connection)
result_2

,PaymentMethod,TotalRevenue
0,Credit Card,263847.63
1,Online,262442.94
2,Cash,259786.29
3,Gift Card,246323.92
4,Debit Card,232361.18


**Finding:** Revenue is fairly evenly distributed across payment methods, ranging from Credit Card at the top (263,847.63) to Debit Card at the bottom (232,361.18), a gap of roughly 12%. No single payment method dominates, and none appears significantly underused, suggesting payment method preference doesn't strongly influence overall spending behavior. This total revenue figure doesn't yet distinguish between a method being used more often versus each transaction being larger, that's examined next.

### 3. What is the average order value overall, and how does it vary by payment method?

Total revenue by payment method showed a fairly even spread, but that figure doesn't separate "more transactions" from "bigger transactions." This section checks the average order value (AOV) to see whether any payment method is associated with larger individual purchases, regardless of how many total orders it processed.

In [8]:
query_3a = """
SELECT AVG(TotalPrice) AS AveragePrice
FROM orders
"""

result_3a = pd.read_sql_query(query_3a, connection)
result_3a

,AveragePrice
0,1053.9683


In [9]:
query_3b = """
SELECT PaymentMethod, AVG(TotalPrice) AS AveragePrice
FROM orders
GROUP BY PaymentMethod
ORDER BY AveragePrice DESC
"""

result_3b = pd.read_sql_query(query_3b, connection)
result_3b

,PaymentMethod,AveragePrice
0,Credit Card,1127.553974
1,Gift Card,1070.973565
2,Cash,1056.041829
3,Online,1017.220698
4,Debit Card,1001.556810


**Finding:** The overall average order value is 1,053.97, sitting roughly in the middle of the per payment method range (1,001.56 to 1,127.55). Credit Card holds the top spot in both total revenue and average order value, while Debit Card sits at the bottom on both. Online and Gift Card tell a more interesting story: Online ranks 2nd in total revenue but drops to 4th in average order value, while Gift Card is the reverse, 4th in total revenue but 2nd in average order value. This suggests Online is used in a higher number of transactions overall, while Gift Card is used less frequently but tends to be associated with higher value purchases when it is used.

## Section 2: Operational Patterns

The second set of questions shifts from revenue toward operational behavior over time and across order outcomes, starting with how revenue trends across the months in the dataset.

### 4. Which months had the highest total revenue?

In [10]:
query_4 = """
SELECT strftime('%Y-%m', Date) AS OrderMonth, SUM(TotalPrice) AS TotalRevenue FROM Orders 
GROUP BY OrderMonth 
ORDER BY TotalRevenue DESC
LIMIT 10
"""

result_4 = pd.read_sql_query(query_4, connection)
result_4

,OrderMonth,TotalRevenue
0,2024-06,68068.54
1,2023-05,63836.84
2,2023-01,56685.75
3,2023-08,54352.14
4,2025-06,53047.40
5,2023-10,52607.85
6,2024-04,49613.14
7,2023-06,49500.19
8,2023-03,48609.37
9,2023-12,43754.73


**Finding:** June appears three times in the top 10 revenue months (2024-06, 2023-06, and 2025-06), with June 2024 generating the highest revenue of any month in the dataset (68,068.54). This suggests a possible seasonal pattern worth monitoring, though with only three years of data, it isn't yet a fully confirmed trend. More notably, 2023 accounts for 6 of the top 10 months, while 2024 and 2025 combined only account for 4, consistent with the gradual decline in order volume identified in Project 2's EDA, revenue appears to have been more heavily concentrated earlier in the dataset's timeframe.

### 5. Which products have the highest number of cancelled orders?

Shifting focus from revenue to operational issues, this section looks at cancellations specifically. Identifying which products are cancelled most often can highlight quality, sizing, or expectation mismatch problems worth investigating further.

In [11]:
query_5 = """
SELECT Product, COUNT(Product) AS ProductCount 
FROM orders
WHERE OrderStatus = 'Cancelled'
GROUP BY Product 
ORDER BY ProductCount DESC
"""

result_5 = pd.read_sql_query(query_5, connection)
result_5

,Product,ProductCount
0,Chair,45
1,Printer,35
2,Monitor,35
3,Laptop,35
4,Desk,35
5,Tablet,34
6,Phone,31


**Finding:** Chair has the highest number of cancelled orders (45), followed closely by Printer, Monitor, Laptop, and Desk (all at 35). Notably, Chair also led total revenue and total order volume from Section 1, so a higher cancellation count isn't entirely surprising given it's simply ordered more often. A rough check confirms this: Chair's cancellation rate is about 25% (45 of 178 total orders), compared to Phone's roughly 20% (31 of 156 orders), a modest elevation rather than a dramatic outlier. Cancellation rate is examined more systematically across categories in the next section.

### 6. Does cancellation rate differ by payment method?

Rather than just counting cancellations, this question calculates cancellation rate, cancelled orders as a percentage of total orders, broken down by payment method. This avoids the trap of confusing a payment method that simply processes more orders overall with one that's genuinely more prone to cancellations.

In [12]:
query_6 = """
SELECT 
    PaymentMethod,
    COUNT(*) AS TotalOrders,
    SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) AS CancelledOrders,
    ROUND(100.0 * SUM(CASE WHEN OrderStatus = 'Cancelled' THEN 1 ELSE 0 END) / COUNT(*), 2) AS CancellationRatePercent
FROM orders
GROUP BY PaymentMethod
ORDER BY CancellationRatePercent DESC
"""

result_6 = pd.read_sql_query(query_6, connection)
result_6

,PaymentMethod,TotalOrders,CancelledOrders,CancellationRatePercent
0,Credit Card,234,54,23.08
1,Gift Card,230,50,21.74
2,Online,258,53,20.54
3,Cash,246,49,19.92
4,Debit Card,232,44,18.97


**Finding:** Cancellation rates across payment methods are tightly clustered, ranging from Credit Card at the high end (23.08%) to Debit Card at the low end (18.97%), a spread of just over 4 percentage points. Credit Card having both the highest revenue and the highest cancellation rate is worth noting, though the gap isn't large enough to call it a clear problem area. Compared to the product level cancellation rates from the previous question (Chair at ~25% vs Phone at ~20%), payment method appears to be an even weaker driver of cancellations than product choice, no single payment method stands out as significantly riskier than the others.

### 7. How many cancelled orders involved a coupon code, versus without one?

This checks whether discounted orders (those with a coupon applied) are more or less likely to end up cancelled compared to orders with no coupon at all. If coupon driven orders cancel at a notably different rate, that could point to lower commitment purchases or impulse buying tied to discount codes.

In [13]:
query_7 = """
SELECT 
    SUM(CASE WHEN CouponCode = 'NO_COUPON' THEN 1 ELSE 0 END) AS CancelledNoCoupon,
    SUM(CASE WHEN CouponCode != 'NO_COUPON' THEN 1 ELSE 0 END) AS CancelledWithCoupon
FROM orders
WHERE OrderStatus = 'Cancelled'
"""

result_7 = pd.read_sql_query(query_7, connection)
result_7

,CancelledNoCoupon,CancelledWithCoupon
0,58,192


In [15]:
query_7a = """
SELECT 
    SUM(CASE WHEN CouponCode = 'NO_COUPON' THEN 1 ELSE 0 END) AS AllOrdersNoCoupon,
    SUM(CASE WHEN CouponCode != 'NO_COUPON' THEN 1 ELSE 0 END) AS AllOrdersWithCoupon
FROM orders
"""

result_7a = pd.read_sql_query(query_7a, connection)
result_7a

,AllOrdersNoCoupon,AllOrdersWithCoupon
0,309,891


**Finding:** Among cancelled orders, 192 had a coupon code applied versus 58 with none. On its own, that split looks significant, but comparing it against the full dataset shows coupon usage sits at 74.25% overall (891 of 1,200 orders), almost identical to the 76.8% rate seen among cancellations specifically. The roughly 2.5 percentage point gap is too small to suggest coupons are meaningfully associated with cancellation behavior, cancelled orders simply reflect the same coupon usage pattern as the dataset as a whole.

## Section 3: Customer Acquisition Insight

The final question shifts focus to marketing effectiveness, not just which referral source brings in the most orders overall, but which one brings in orders that actually convert into successful deliveries. A source that drives high order volume but low successful delivery isn't as valuable as one with fewer orders but a stronger completion rate.

### 8. Which referral source brings in the most successfully delivered orders?

In [17]:
query_8 = """
SELECT ReferralSource, COUNT(OrderID) AS DeliveredOrderCount 
FROM orders 
WHERE OrderStatus = 'Delivered' 
GROUP BY ReferralSource 
ORDER by DeliveredOrderCount DESC
"""

result_8 = pd.read_sql_query(query_8, connection)
result_8

,ReferralSource,DeliveredOrderCount
0,Email,60
1,Instagram,52
2,Google,44
3,Facebook,42
4,Referral,33


In [18]:
query_8b = """
SELECT 
    ReferralSource,
    COUNT(*) AS TotalOrders,
    SUM(CASE WHEN OrderStatus = 'Delivered' THEN 1 ELSE 0 END) AS DeliveredOrders,
    ROUND(100.0 * SUM(CASE WHEN OrderStatus = 'Delivered' THEN 1 ELSE 0 END) / COUNT(*), 2) AS DeliveryRatePercent
FROM orders
GROUP BY ReferralSource
ORDER BY DeliveryRatePercent DESC
"""

result_8b = pd.read_sql_query(query_8b, connection)
result_8b

,ReferralSource,TotalOrders,DeliveredOrders,DeliveryRatePercent
0,Email,250,60,24.00
1,Instagram,259,52,20.08
2,Facebook,228,42,18.42
3,Google,241,44,18.26
4,Referral,222,33,14.86


**Finding:** While Instagram drives slightly more total orders (259 vs Email's 250), Email has the highest delivery rate at 24.00%, compared to Instagram's 20.08%, a meaningful gap of nearly 4 percentage points. This suggests Email referrals convert more reliably into completed deliveries, even though Instagram generates marginally more raw traffic. Referral stands out as the weakest channel on both measures, the lowest total orders (222) and the lowest delivery rate (14.86%), making it the clearest underperformer among all referral sources.

## Conclusion

This project moved beyond viewing the cleaned dataset in pandas and instead queried it directly through SQL, using a SQLite database built from the cleaned order records. Working through eight business questions surfaced a few key patterns worth highlighting:

**Revenue is broad based, not concentrated.** Chair leads in total revenue, but not because of a high unit price or even the most orders, it wins because customers tend to buy more units of it per transaction than any other product. Payment methods show a similarly even spread, no single method dominates total revenue or average order value by a wide margin.

**Operational issues exist but aren't alarming.** Cancellation rates vary modestly by product and by payment method, Chair and Credit Card sit slightly higher than their peers, but the gaps are too small to call either a clear problem area. Coupon usage among cancelled orders closely mirrors coupon usage across the dataset as a whole, ruling out coupons as a meaningful driver of cancellation behavior.

**Raw volume and quality of outcome are not the same thing.** Instagram brings in the most referral traffic, but Email converts that traffic into successfully delivered orders at a notably higher rate, 24.00% versus 20.08%. Referral, meanwhile, underperforms on both volume and conversion, making it the weakest acquisition channel in the dataset.

Across all three sections, the consistent lesson was the same one the Project 3 brief opened with: a single number rarely tells the full story. Every finding in this notebook that held up under scrutiny did so because it was checked against a baseline or a second metric, total revenue against order count, cancellation count against cancellation rate, referral volume against referral conversion. Where a pattern didn't hold up to that check, like coupon usage among cancellations, it was reported as a non finding rather than overstated.